In [0]:
%sql
-- =============================================================================
-- Notebook: 05_nb_silver_scd2_regional_sales_targets_scj
-- Purpose: Slowly Changing Dimension Type 2 implementation for regional_sales_targets
-- Pattern: SCJ Databricks Lakehouse - Silver Layer
-- Source: nestle_dev_bronze.sharepoint.regional_sales_targets
-- Target: nestle_dev_silver.core.regional_sales_targets_scd2
-- =============================================================================

-- === STEP 1: Ensure SCD2 table exists ===
-- One-time setup: creates the target table structure if it doesn't exist
-- Safe to run repeatedly (idempotent)

CREATE TABLE IF NOT EXISTS nestle_dev_silver.core.regional_sales_targets_scd2 (
  product_id STRING,
  region STRING,
  period STRING,
  target_amount DECIMAL(18,2),
  target_category STRING,
  dbt_valid_from DATE,
  dbt_valid_to DATE,
  dbt_is_current BOOLEAN
);

-- === STEP 2: Expire rows with changed values ===
-- Logic: Find rows in Silver that are current (is_current=TRUE) but whose
-- business key matches a Bronze row with a DIFFERENT target_amount.
-- Mark those Silver rows as expired (is_current=FALSE, valid_to=today).

UPDATE nestle_dev_silver.core.regional_sales_targets_scd2 AS target
SET dbt_valid_to = current_date(),
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1
    FROM nestle_dev_bronze.sharepoint.regional_sales_targets AS src
    WHERE src.product_id = target.product_id
      AND src.region = target.region
      AND src.period = target.period
      AND src.target_amount <> target.target_amount
  );

-- === STEP 3: Insert new/current rows ===
-- Logic: For every Bronze row, check if Silver already has a CURRENT row for that key.
-- If not (either never existed, or was just expired), insert a fresh current row.

INSERT INTO nestle_dev_silver.core.regional_sales_targets_scd2
SELECT
  src.product_id,
  src.region,
  src.period,
  CAST(src.target_amount AS DECIMAL(18,2)) AS target_amount,
  CASE
    WHEN src.target_amount < 5000 THEN 'Low'
    WHEN src.target_amount < 10000 THEN 'Medium'
    ELSE 'High'
  END AS target_category,
  current_date() AS dbt_valid_from,
  CAST('2099-12-31' AS DATE) AS dbt_valid_to,
  TRUE AS dbt_is_current
FROM nestle_dev_bronze.sharepoint.regional_sales_targets AS src
WHERE NOT EXISTS (
  SELECT 1
  FROM nestle_dev_silver.core.regional_sales_targets_scd2 AS tgt
  WHERE tgt.product_id = src.product_id
    AND tgt.region = src.region
    AND tgt.period = src.period
    AND tgt.dbt_is_current = TRUE
);

-- === STEP 4: Audit Log Entry ===
-- Record this SCD2 run in the control audit table for traceability

INSERT INTO nestle_dev_silver.control.ingestion_audit_log (
  source_id,
  job_id,
  run_id,
  source_system_name,
  source_dataset_name,
  ingestion_timestamp,
  rows_read,
  rows_written,
  watermark_value,
  status,
  error_details,
  pipeline_name
)
SELECT
  'sharepoint_regional_targets' AS source_id,
  NULL AS job_id,
  NULL AS run_id,
  'sharepoint' AS source_system_name,
  'regional_sales_targets' AS source_dataset_name,
  current_timestamp() AS ingestion_timestamp,
  COUNT(DISTINCT CONCAT(product_id, '|', region, '|', period)) AS rows_read,
  COUNT(*) AS rows_written,
  CAST(current_date() AS STRING) AS watermark_value,
  'SUCCESS' AS status,
  NULL AS error_details,
  '05_nb_silver_scd2_regional_sales_targets_scj' AS pipeline_name
FROM nestle_dev_silver.core.regional_sales_targets_scd2
WHERE dbt_valid_from = current_date();

-- === STEP 5: Verification Query ===
-- Show the current state: all current rows and a count of historical versions

SELECT
  product_id,
  region,
  period,
  target_amount,
  target_category,
  dbt_valid_from,
  dbt_valid_to,
  dbt_is_current,
  COUNT(*) OVER (PARTITION BY product_id, region, period) AS version_count
FROM nestle_dev_silver.core.regional_sales_targets_scd2
ORDER BY product_id, region, period, dbt_valid_from;